In [1]:
# STEP 1 : Load + preprocess Titanic data

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# A) Load dataset from Kaggle input

# Common Kaggle Titanic competition path:
default_path = "/kaggle/input/competitions/titanic/train.csv"

if os.path.exists(default_path):
    data_path = default_path
else:
    # Fallback: auto-search for train.csv in /kaggle/input
    found = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower() == "train.csv":
                found.append(os.path.join(root, f))
    if not found:
        raise FileNotFoundError("Could not find train.csv under /kaggle/input. Please check attached dataset.")
    data_path = found[0]

df = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Original shape:", df.shape)

# B) Keep required columns

# Target + useful predictors (includes categorical: Sex, Embarked)
required_cols = ["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

df = df[required_cols].copy()

# C) Handle missing values

# Numeric
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# Categorical
# Fill Embarked with mode
if df["Embarked"].isna().any():
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# D) Preprocess categorical features

# 1) Sex -> binary numeric
# female=1, male=0 (simple and explicit)
df["Sex"] = df["Sex"].map({"male": 0, "female": 1}).astype(np.float32)

# Safety check: no unmapped categories
if df["Sex"].isna().any():
    raise ValueError("Unexpected values in 'Sex' column after mapping.")

# 2) Embarked -> one-hot encoding
# drop_first=True avoids perfect multicollinearity in logistic regression
emb_dummies = pd.get_dummies(df["Embarked"], prefix="Embarked", drop_first=True, dtype=np.float32)
df = pd.concat([df.drop(columns=["Embarked"]), emb_dummies], axis=1)

# E) Build X, y

y = df["Survived"].astype(np.float32).values
X = df.drop(columns=["Survived"]).astype(np.float32).values
feature_names = df.drop(columns=["Survived"]).columns.tolist()

# F) Train/validation split (stratified for balanced class ratio)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# G) Standardize features (recommended for stable gradient descent)

x_mean = X_train.mean(axis=0, keepdims=True)
x_std = X_train.std(axis=0, keepdims=True) + 1e-8

X_train = (X_train - x_mean) / x_std
X_val = (X_val - x_mean) / x_std

# H) Final checks

print("\nAfter preprocessing:")
print("Train shape:", X_train.shape, "Val shape:", X_val.shape)
print("Number of features:", len(feature_names))
print("Feature names:", feature_names)

print("\nNull checks:")
print("Any NaN in X_train?", np.isnan(X_train).any())
print("Any NaN in X_val?", np.isnan(X_val).any())
print("Any NaN in y_train?", np.isnan(y_train).any())
print("Any NaN in y_val?", np.isnan(y_val).any())

print("\nClass distribution:")
print("Train survival rate:", y_train.mean())
print("Val survival rate:", y_val.mean())

Loaded: /kaggle/input/competitions/titanic/train.csv
Original shape: (891, 12)

After preprocessing:
Train shape: (712, 8) Val shape: (179, 8)
Number of features: 8
Feature names: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']

Null checks:
Any NaN in X_train? False
Any NaN in X_val? False
Any NaN in y_train? False
Any NaN in y_val? False

Class distribution:
Train survival rate: 0.38342696
Val survival rate: 0.38547486
